# ESG Supply Chain Risk Intelligence – Case Study (Simulated Data)

This notebook builds an end-to-end **responsible procurement risk** dataset and analytics workflow:
- Data consolidation (suppliers, ESG scores, audits, incidents)
- Data quality checks + light imputation for scoring
- Composite ESG risk score and **risk × spend** prioritization
- Export of curated tables for Tableau dashboards

> **Note:** Data is fully simulated for portfolio purposes.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../data")
OUT_DIR = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

suppliers = pd.read_csv(DATA_DIR/"suppliers.csv")
esg_scores = pd.read_csv(DATA_DIR/"esg_scores.csv")
audits = pd.read_csv(DATA_DIR/"audits.csv")
incidents = pd.read_csv(DATA_DIR/"incidents.csv")

suppliers.shape, esg_scores.shape, audits.shape, incidents.shape


## 1) Merge and basic profiling

In [ ]:
# Aggregate incidents to supplier-level penalties
sev_map = {"Low":3,"Medium":7,"High":12}
inc = incidents.copy()
inc["sev_penalty"] = inc["severity"].map(sev_map)
inc_pen = inc.groupby("supplier_id")["sev_penalty"].sum().reset_index().rename(columns={"sev_penalty":"incident_penalty"})

df = (suppliers
      .merge(esg_scores, on="supplier_id", how="left")
      .merge(audits, on="supplier_id", how="left")
      .merge(inc_pen, on="supplier_id", how="left"))

df["incident_penalty"] = df["incident_penalty"].fillna(0)

df.head()


In [ ]:
# Missingness snapshot
missing = df[["env_score","social_score","governance_score"]].isna().mean().sort_values(ascending=False)
missing


## 2) Data quality + imputation (for scoring only)

In [ ]:
df["esg_fields_present"] = df[["env_score","social_score","governance_score"]].notna().sum(axis=1)
df["esg_completeness"] = df["esg_fields_present"] / 3.0

for col in ["env_score","social_score","governance_score"]:
    df[col+"_imputed"] = df[col]
    med_by_cat = df.groupby("category")[col].transform("median")
    overall_med = df[col].median()
    df[col+"_imputed"] = df[col+"_imputed"].fillna(med_by_cat).fillna(overall_med)

df[["supplier_id","env_score","env_score_imputed","esg_completeness"]].head()


## 3) Risk scoring model (transparent & explainable)

In [ ]:
# Penalties (explainable business rules)
df["dependency_penalty"] = df["dependency_level"].map({"Low":0,"Medium":2,"High":5})
df["tier_penalty"] = df["tier"].map({"Tier 1":0,"Tier 2":4,"Tier 3":8})
df["traceability_penalty"] = df["traceability_level"].map({"Full":0,"Partial":4,"Limited":10})
df["charter_penalty"] = np.where(df["charter_signed"], 0, 12)
df["made_in_penalty"] = np.where(df["made_in_disclosed"], 0, 5)

def audit_penalty(row):
    if not bool(row.get("audited", False)):
        return 6 if row["country_risk_level"] in ("High","Medium") else 2
    pen = 0
    if row.get("non_compliance_level","") == "Major":
        pen += 15
    elif row.get("non_compliance_level","") == "Minor":
        pen += 7
    if pd.notna(row.get("audit_score", np.nan)):
        pen += max(0, (70 - float(row["audit_score"])) * 0.25)
    return pen

df["audit_penalty"] = df.apply(audit_penalty, axis=1)

# Weighted risk score (social pillar weighted higher)
df["risk_score"] = (
    (100 - df["env_score_imputed"]) * 0.28 +
    (100 - df["social_score_imputed"]) * 0.40 +
    (100 - df["governance_score_imputed"]) * 0.32 +
    df["country_risk_weight"]*0.55 +
    df["dependency_penalty"] +
    df["tier_penalty"] +
    df["traceability_penalty"] +
    df["charter_penalty"] +
    df["made_in_penalty"] +
    df["incident_penalty"]*0.55 +
    df["audit_penalty"]
).clip(0, 100)

def classify(score):
    if score >= 70: return "High Risk"
    if score >= 40: return "Medium Risk"
    return "Low Risk"

df["risk_class"] = df["risk_score"].apply(classify)

df["risk_class"].value_counts()


## 4) Quick visuals (portfolio-ready)

In [ ]:
# Risk distribution
plt.figure()
df["risk_class"].value_counts().plot(kind="bar")
plt.title("Suppliers by risk class (simulated)")
plt.xlabel("Risk class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Spend exposure by risk class
plt.figure()
(df.groupby("risk_class")["annual_spend_chf"].sum().sort_values(ascending=False)/1e6).plot(kind="bar")
plt.title("Annual spend exposure by risk class (CHF, millions)")
plt.xlabel("Risk class")
plt.ylabel("CHF (millions)")
plt.tight_layout()
plt.show()

# Risk vs spend matrix (scatter)
plt.figure()
plt.scatter(df["annual_spend_chf"], df["risk_score"], alpha=0.35)
plt.xscale("log")
plt.title("Risk × Spend prioritization matrix (log spend)")
plt.xlabel("Annual spend (CHF, log scale)")
plt.ylabel("Risk score")
plt.tight_layout()
plt.show()


## 5) Export curated tables for Tableau

In [ ]:
# Recommended actions (simple rules)
def recommend(row):
    if row.get("non_compliance_level","") == "Major":
        return "Immediate corrective action + 30-day follow-up"
    if not bool(row["charter_signed"]):
        return "Request signature/commitment before continuation"
    if row["risk_class"] == "High Risk":
        return "Prioritize for audit and mitigation plan"
    if row["risk_class"] == "Medium Risk":
        return "Monitor quarterly; request additional evidence"
    return "Approved; monitor annually"

df["recommended_action"] = df.apply(recommend, axis=1)

# Exports
df.to_csv(OUT_DIR/"cleaned_master_dataset.csv", index=False)

(df[["supplier_id","supplier_name","category","country","region","annual_spend_chf",
     "risk_score","risk_class","recommended_action","esg_completeness","data_source","last_update"]]
   .to_csv(OUT_DIR/"risk_scores.csv", index=False))

OUT_DIR/"risk_scores.csv"
